In [2]:
pip install groq datasets scikit-learn numpy pandas tqdm radon

  Using cached groq-1.0.0-py3-none-any.whl (138 kB)
  Using cached datasets-4.6.1-py3-none-any.whl (520 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 1.1/1.1 MB 8.9 MB/s eta 0:00:00
     ------------------------------------- 520.7/520.7 kB 31.9 MB/s eta 0:00:00
  Using cached pandas-2.3.3-cp310-cp310-win_amd64.whl (11.3 MB)
     ---------------------------------------- 52.8/52.8 kB ? eta 0:00:00
  Using cached jiter-0.13.0-cp310-cp310-win_amd64.whl (206 kB)
     --------------------------------------- 27.5/27.5 MB 32.8 MB/s eta 0:00:00
     ---------------------------------------- 119.7/119.7 kB ? eta 0:00:00
     -------------------------------------- 134.9/134.9 kB 2.7 MB/s eta 0:00:00
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)
  Using cached aiohttp-3.13.3-cp310-cp310-win_amd64.whl (456 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl (6.2 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Using cached attrs-25.4.0-py3-none-any.whl 


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install google-genai


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
pip uninstall torch torchvision torchaudio -y

Found existing installation: torch 2.10.0
Uninstalling torch-2.10.0:
  Successfully uninstalled torch-2.10.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 2.4/2.4 GB 856.5 kB/s eta 0:00:00
     ---------------------------------------- 6.1/6.1 MB 42.9 MB/s eta 0:00:00
     ---------------------------------------- 4.1/4.1 MB 65.4 MB/s eta 0:00:00
  Obtaining dependency information for sympy==1.13.1 from https://files.pythonhosted.org/packages/b2/fe/81695a1aa331a842b582453b605175f419fe8540355886031328089d840a/sympy-1.13.1-py3-none-any.whl.metadata
  Obtaining dependency information for pillow!=8.3.*,>=5.3.0 from https://files.pythonhosted.org/packages/92/c6/c2f2fc7e56301c21827e689bb8b0b465f1b52878b57471a070678c0c33cd/pillow-12.0.0-cp310-cp310-win_amd64.whl.metadata
   ---------------------------------------- 6.2/6.2 MB 18.8 MB/s eta 0:00:00
Using cached pillow-12.0.0-cp310-cp310-win_amd64.whl (7.0 MB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached pillow-12.0.0-cp310-cp310-win_amd64.whl (7.0 MB)
  Attempting uninstall: 


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Torch CUDA version:", torch.version.cuda)

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Torch CUDA version: 12.1


In [3]:
!pip install datasets transformers torch pandas tqdm radon numpy accelerate

  Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:



import torch
import ast
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from radon.complexity import cc_visit
import torch.nn.functional as F


MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SAMPLES = 160
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

model.eval()

print("Model device:", next(model.parameters()).device)


dataset = load_dataset("openai_humaneval")["test"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))


def generate_with_logprobs(prompt):
    
    full_prompt = prompt + "\n"

    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs.sequences[0]
    new_tokens = generated_tokens[inputs.input_ids.shape[1]:]

    logprobs = []
    for i, score in enumerate(outputs.scores):
        probs = F.log_softmax(score, dim=-1)
        token_id = new_tokens[i]
        logprobs.append(probs[0, token_id].item())

    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    
    if "```" in generated_text:
        generated_text = generated_text.split("```")[1]
  
    final_code = prompt + generated_text

    return final_code, mean_logprob


def compute_confidence(mean_logprob):
    return float(1 / (1 + np.exp(-mean_logprob)))


def run_tests(code, test_code):
    fname = "temp_test.py"

    with open(fname, "w", encoding="utf-8") as f:
        f.write(code + "\n\n")
        f.write(test_code)

    try:
        subprocess.check_output(["python", fname], timeout=5)
        return 0
    except:
        return 1

def extract_features(prompt, code, mean_logprob):
    features = {}

    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    try:
        tree = ast.parse(code)
        features["ast_nodes"] = len(list(ast.walk(tree)))
    except:
        features["ast_nodes"] = 0

    try:
        complexity = cc_visit(code)
        features["avg_complexity"] = (
            sum(c.complexity for c in complexity) / len(complexity)
            if complexity else 0
        )
    except:
        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)

    return features

records = []

for item in tqdm(dataset):
    prompt = item["prompt"]
    tests = item["test"]

    try:
        code, mean_logprob = generate_with_logprobs(prompt)
    except:
        code = ""
        mean_logprob = -100

    label = run_tests(code, tests)

    feats = extract_features(prompt, code, mean_logprob)
    feats["failed"] = label

    records.append(feats)

df = pd.DataFrame(records)
df.to_csv("failure_risk_dataset.csv", index=False)

print("Dataset shape:", df.shape)
df.head()

In [4]:
dataset = load_dataset("openai_humaneval")["test"]

len(dataset)

164

Model Loading

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32
).to(DEVICE)

model.eval()

print("Model device:", next(model.parameters()).device)
print("Model loaded successfully.")

c:\Users\VINIL\Desktop\Failure_Aware_Agent\Environment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA available: True
Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


Loading weights: 100%|██████████| 434/434 [00:04<00:00, 105.34it/s, Materializing param=model.norm.weight]                              


Model device: cuda:0
Model loaded successfully.


HumanEval

In [2]:
import ast
import subprocess
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from radon.complexity import cc_visit
import torch.nn.functional as F

MAX_SAMPLES = 160

dataset = load_dataset("openai_humaneval")["test"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

def generate_with_logprobs(prompt):

    full_prompt = prompt + "\n"
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs.sequences[0]
    new_tokens = generated_tokens[inputs.input_ids.shape[1]:]

    logprobs = []
    for i, score in enumerate(outputs.scores):
        probs = F.log_softmax(score, dim=-1)
        token_id = new_tokens[i]
        logprobs.append(probs[0, token_id].item())

    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    if "```" in generated_text:
        generated_text = generated_text.split("```")[1]

    final_code = prompt + generated_text

    return final_code, mean_logprob


def compute_confidence(mean_logprob):
    return float(1 / (1 + np.exp(-mean_logprob)))


def run_tests(code, test_code):

    fname = "temp_test.py"

    with open(fname, "w", encoding="utf-8") as f:
        f.write(code + "\n\n")
        f.write(test_code)

    try:
        subprocess.check_output(["python", fname], timeout=5)
        return 0
    except:
        return 1


def extract_features(prompt, code, mean_logprob):

    features = {}

    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    try:
        tree = ast.parse(code)
        features["ast_nodes"] = len(list(ast.walk(tree)))
    except:
        features["ast_nodes"] = 0

    try:
        complexity = cc_visit(code)
        features["avg_complexity"] = (
            sum(c.complexity for c in complexity) / len(complexity)
            if complexity else 0
        )
    except:
        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)

    return features


records = []

for item in tqdm(dataset):

    prompt = item["prompt"]
    tests = item["test"]

    try:
        code, mean_logprob = generate_with_logprobs(prompt)
    except:
        code = ""
        mean_logprob = -100

    label = run_tests(code, tests)

    feats = extract_features(prompt, code, mean_logprob)
    feats["failed"] = label

    records.append(feats)


df = pd.DataFrame(records)
df.to_csv("failure_risk_dataset.csv", index=False)

print("Dataset shape:", df.shape)
print(df.head())

100%|██████████| 160/160 [18:27<00:00,  6.92s/it]

Dataset shape: (160, 7)
   prompt_len  code_len  ast_nodes  avg_complexity  mean_logprob  confidence  \
0         348       600         86             4.0     -0.108508    0.472900   
1         506       930         98             5.0     -0.151814    0.462119   
2         331       743          0             0.0     -0.059094    0.485231   
3         448       663         55             3.0     -0.104605    0.473873   
4         430       821         81             2.0     -0.166772    0.458403   

   failed  
0       0  
1       0  
2       1  
3       0  
4       0  


In [5]:
import pandas as pd

df = pd.read_csv("failure_risk_dataset.csv")

print(df["failed"].value_counts())

failed
1    103
0     57
Name: count, dtype: int64


MBPP dataset



In [6]:
from datasets import load_dataset
dataset = load_dataset("google-research-datasets/mbpp")["test"]
len(dataset)

Generating prompt split: 100%|██████████| 10/10 [00:00<00:00, 745.24 examples/s]


500

In [12]:
import ast
import subprocess
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from radon.complexity import cc_visit
import torch.nn.functional as F
import re

MAX_SAMPLES = 500

dataset = load_dataset("google-research-datasets/mbpp")["test"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))


def extract_function_name(reference_code):
    try:
        return reference_code.split("def ")[1].split("(")[0]
    except:
        return "solution"


def clean_generated_code(text):

    text = text.replace("```python", "").replace("```", "")

    pattern = r"def\s+\w+\(.*?\):[\s\S]*"
    match = re.search(pattern, text)

    if match:
        return match.group(0)

    return text.strip()


def generate_with_logprobs(prompt, func_name):

    full_prompt = f"""
You are a Python coding assistant.

Write a Python function named `{func_name}` that solves the problem.

Problem:
{prompt}

Return ONLY valid Python code.
No explanation.
"""

    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs.sequences[0]
    new_tokens = generated_tokens[inputs.input_ids.shape[1]:]

    logprobs = []

    for i, score in enumerate(outputs.scores):
        probs = F.log_softmax(score, dim=-1)
        token_id = new_tokens[i]
        logprobs.append(probs[0, token_id].item())

    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    generated_text = clean_generated_code(generated_text)

    return generated_text, mean_logprob


def compute_confidence(mean_logprob):
    return float(1 / (1 + np.exp(-mean_logprob)))


def run_tests(code, test_list):

    fname = "temp_test.py"

    tests = "\n".join(["    " + t for t in test_list])

    script = f"""
{code}

if __name__ == "__main__":
{tests}
"""

    with open(fname, "w", encoding="utf-8") as f:
        f.write(script)

    try:
        subprocess.check_output(["python", fname], timeout=5)
        return 0
    except subprocess.CalledProcessError:
        return 1
    except Exception:
        return 1


def extract_features(prompt, code, mean_logprob):

    features = {}

    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    try:
        tree = ast.parse(code)
        features["ast_nodes"] = len(list(ast.walk(tree)))
    except:
        features["ast_nodes"] = 0

    try:
        complexity = cc_visit(code)
        features["avg_complexity"] = (
            sum(c.complexity for c in complexity) / len(complexity)
            if complexity else 0
        )
    except:
        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)

    return features


records = []

for item in tqdm(dataset):

    prompt = item["text"]
    tests = item["test_list"]

    func_name = extract_function_name(item["code"])

    try:
        code, mean_logprob = generate_with_logprobs(prompt, func_name)
    except:
        code = ""
        mean_logprob = -100

    label = run_tests(code, tests)

    feats = extract_features(prompt, code, mean_logprob)
    feats["failed"] = label

    records.append(feats)


df = pd.DataFrame(records)

df.to_csv("failure_risk_dataset_mbpp.csv", index=False)

print("Dataset shape:", df.shape)
print(df.head())

100%|██████████| 500/500 [37:51<00:00,  4.54s/it]  

Dataset shape: (500, 7)
   prompt_len  code_len  ast_nodes  avg_complexity  mean_logprob  confidence  \
0          97        53         15             1.0     -0.104610    0.473871   
1          92        60         13             1.0     -0.092184    0.476970   
2          64      1002          0             0.0     -0.194537    0.451519   
3          65      1023          0             0.0     -0.066801    0.483306   
4          56        80         20             3.0     -0.088981    0.477770   

   failed  
0       0  
1       0  
2       1  
3       1  
4       1  


In [13]:
import pandas as pd

df = pd.read_csv("failure_risk_dataset_mbpp.csv")

print(df["failed"].value_counts())

failed
1    327
0    173
Name: count, dtype: int64


code search net

In [16]:
!pip install -U datasets


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from datasets import load_dataset
dataset = load_dataset("code_search_net", "python")["train"]
len(dataset)
dataset[0]

{'repository_name': 'mjirik/imcut',
 'func_path_in_repository': 'imcut/pycut.py',
 'func_name': 'ImageGraphCut.__msgc_step3_discontinuity_localization',
 'whole_func_string': 'def __msgc_step3_discontinuity_localization(self):\n        """\n        Estimate discontinuity in basis of low resolution image segmentation.\n        :return: discontinuity in low resolution\n        """\n        import scipy\n\n        start = self._start_time\n        seg = 1 - self.segmentation.astype(np.int8)\n        self.stats["low level object voxels"] = np.sum(seg)\n        self.stats["low level image voxels"] = np.prod(seg.shape)\n        # in seg is now stored low resolution segmentation\n        # back to normal parameters\n        # step 2: discontinuity localization\n        # self.segparams = sparams_hi\n        seg_border = scipy.ndimage.filters.laplace(seg, mode="constant")\n        logger.debug("seg_border: %s", scipy.stats.describe(seg_border, axis=None))\n        # logger.debug(str(np.max(seg

In [2]:

import ast
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from radon.complexity import cc_visit
import torch.nn.functional as F
import re

# Disable gradients for speed
torch.set_grad_enabled(False)

MAX_SAMPLES = 500
MAX_PROMPT_LEN = 400
MAX_NEW_TOKENS = 800

# Load dataset
dataset = load_dataset("code_search_net", "python")["train"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

# Precompiled regex
FUNC_PATTERN = re.compile(r"def\s+\w+\(.*?\):[\s\S]*")


def extract_function_name(code):
    try:
        return code.split("def ")[1].split("(")[0]
    except:
        return "solution"


def clean_generated_code(text):

    text = text.replace("```python", "").replace("```", "")
    match = FUNC_PATTERN.search(text)

    if match:
        return match.group(0)

    return text.strip()


def generate_with_logprobs(prompt, func_name):

    prompt = prompt[:MAX_PROMPT_LEN]

    full_prompt = f"""
Write a Python function named `{func_name}`.

Problem:
{prompt}

Return only valid Python code.
"""

    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs.sequences[0]
    new_tokens = generated_tokens[inputs.input_ids.shape[1]:]

    logprobs = []

    for i, score in enumerate(outputs.scores):
        probs = F.log_softmax(score, dim=-1)
        logprobs.append(probs[0, new_tokens[i]].item())

    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    generated_text = clean_generated_code(generated_text)

    return generated_text, mean_logprob


def compute_confidence(mean_logprob):
    return float(np.exp(mean_logprob))


def check_syntax(code):

    if not code.strip():
        return 1

    try:
        compile(code, "<string>", "exec")
        return 0
    except:
        return 1


def extract_features(prompt, code, mean_logprob):

    features = {}

    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    try:
        tree = ast.parse(code)
        features["ast_nodes"] = sum(1 for _ in ast.walk(tree))
    except:
        features["ast_nodes"] = 0

    try:
        complexity = cc_visit(code)
        features["avg_complexity"] = (
            sum(c.complexity for c in complexity) / len(complexity)
            if complexity else 0
        )
    except:
        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)

    return features


records = []

skipped_samples = 0
skipped_long_prompt = 0
skipped_generation_error = 0


for i, item in enumerate(tqdm(dataset)):

    prompt = item["func_documentation_string"]

    # Skip very long prompts
    if len(prompt) > 600:
        skipped_samples += 1
        skipped_long_prompt += 1
        print(f"⚠️ Skipped sample {i} (prompt too long)")
        continue

    reference_code = item["func_code_string"]
    func_name = extract_function_name(reference_code)

    try:
        code, mean_logprob = generate_with_logprobs(prompt, func_name)
    except:
        skipped_samples += 1
        skipped_generation_error += 1
        print(f"⚠️ Skipped sample {i} (generation failed)")
        continue

    label = check_syntax(code)

    feats = extract_features(prompt, code, mean_logprob)
    feats["failed"] = label

    records.append(feats)


df = pd.DataFrame(records)

df.to_csv("failure_risk_dataset_codesearchnet.csv", index=False)

print("\nDataset shape:", df.shape)
print("Total skipped samples:", skipped_samples)
print("Skipped due to long prompts:", skipped_long_prompt)
print("Skipped due to generation errors:", skipped_generation_error)

print("\nSample rows:")
print(df.head())


  9%|▊         | 43/500 [23:41<3:22:42, 26.61s/it]

⚠️ Skipped sample 43 (prompt too long)
⚠️ Skipped sample 44 (prompt too long)


  9%|▉         | 46/500 [23:51<1:42:04, 13.49s/it]

⚠️ Skipped sample 46 (prompt too long)


 11%|█▏        | 57/500 [27:01<2:30:25, 20.37s/it]

...

⚠️ Skipped sample 408 (prompt too long)


 98%|█████████▊| 488/500 [2:17:29<02:53, 14.43s/it]

⚠️ Skipped sample 488 (prompt too long)


100%|██████████| 500/500 [2:20:52<00:00, 16.90s/it]


Dataset shape: (476, 7)
Total skipped samples: 24
Skipped due to long prompts: 24
Skipped due to generation errors: 0

Sample rows:
   prompt_len  code_len  ast_nodes  avg_complexity  mean_logprob  confidence  \
0         118      1251          0             0.0     -0.250868    0.778125   
1         305      3971          0             0.0     -0.142013    0.867610   
2         307      3854          0             0.0     -0.212942    0.808203   
3         338      1570          0             0.0     -0.189090    0.827712   
4         350      1763          0             0.0     -0.129143    0.878848   

   failed  
0       1  
1       1  
2       1  
3       1  
4       1  


In [ ]:
import pandas as pd

df = pd.read_csv("failure_risk_dataset_mbpp.csv")

print(df["failed"].value_counts())

In [2]:
!pip install sentence-transformers faiss-cpu


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ==============================
# INSTALL FIRST
# ==============================
# pip install datasets tqdm radon sentence-transformers faiss-cpu pandas numpy torch

import ast
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from radon.complexity import cc_visit
import torch.nn.functional as F
import re
from sentence_transformers import SentenceTransformer
import faiss
import os
import json

torch.set_grad_enabled(False)

MAX_SAMPLES = 10
MAX_PROMPT_LEN = 400
MAX_NEW_TOKENS = 1000
TOP_K_HISTORY = 2

CSV_FILE = "failure_risk_dataset.csv"
FAISS_FILE = "history_index.faiss"
LABEL_FILE = "history_labels.npy"

dataset = load_dataset("code_search_net", "python")["train"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dim = 384

# ==============================
# LOAD OR CREATE VECTOR DB
# ==============================

if os.path.exists(FAISS_FILE):

    faiss_index = faiss.read_index(FAISS_FILE)
    history_labels = list(np.load(LABEL_FILE))

else:

    faiss_index = faiss.IndexFlatL2(embedding_dim)
    history_labels = []

# ==============================
# CODE CLEANING
# ==============================

def clean_code(code):

    code = code.replace("```python","").replace("```","").strip()

    return code


# ==============================
# GENERATION
# ==============================

def generate_reasoning_code(prompt, func_name):

    prompt = prompt[:MAX_PROMPT_LEN]

    full_prompt = f"""
Solve the programming problem.

Return ONLY valid JSON.

{{
 "reasoning": "short explanation of algorithm",
 "code": "complete python function"
}}

Problem:
{prompt}

Function name: {func_name}
"""

    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs.sequences[0]
    new_tokens = generated_tokens[inputs.input_ids.shape[1]:]

    logprobs = []

    for i, score in enumerate(outputs.scores):

        probs = F.log_softmax(score, dim=-1)
        logprobs.append(probs[0,new_tokens[i]].item())

    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    text = tokenizer.decode(new_tokens, skip_special_tokens=True)

    reasoning = ""
    code = ""

    try:

        data = json.loads(text)

        reasoning = data["reasoning"]
        code = data["code"]

    except:

        # fallback parser
        reasoning = ""
        code = text

    code = clean_code(code)

    return reasoning.strip(), code.strip(), mean_logprob


# ==============================
# FEATURES
# ==============================

def compute_confidence(mean_logprob):

    return float(np.exp(mean_logprob))


def check_syntax(code):

    if not code.strip():
        return 1

    try:

        compile(code,"<string>","exec")
        return 0

    except:

        return 1


def compute_prior_history(embedding):

    if len(history_labels) < TOP_K_HISTORY:

        return 0.5

    query = np.array([embedding]).astype("float32")

    D,I = faiss_index.search(query,TOP_K_HISTORY)

    sims = 1/(1+D[0])

    labels = np.array(history_labels)[I[0]]

    prior = np.sum(sims*labels)/np.sum(sims)

    return float(prior)


def extract_features(prompt, code, mean_logprob, prior):

    features = {}

    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    try:

        tree = ast.parse(code)
        features["ast_nodes"] = sum(1 for _ in ast.walk(tree))

    except:

        features["ast_nodes"] = 0

    try:

        complexity = cc_visit(code)

        features["avg_complexity"] = (
            sum(c.complexity for c in complexity)/len(complexity)
            if complexity else 0
        )

    except:

        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)
    features["prior_history"] = prior

    return features


# ==============================
# CSV INIT
# ==============================

if not os.path.exists(CSV_FILE):

    cols = [
        "prompt_len",
        "code_len",
        "ast_nodes",
        "avg_complexity",
        "mean_logprob",
        "confidence",
        "prior_history",
        "failed"
    ]

    pd.DataFrame(columns=cols).to_csv(CSV_FILE,index=False)


# ==============================
# MAIN LOOP
# ==============================

skipped = 0

for i,item in enumerate(tqdm(dataset)):

    prompt = item["func_documentation_string"]

    if len(prompt)>600:

        skipped+=1
        continue

    reference_code = item["func_code_string"]

    try:

        func_name = reference_code.split("def ")[1].split("(")[0]

    except:

        func_name="solution"

    try:

        reasoning,code,mean_logprob = generate_reasoning_code(prompt,func_name)

    except:

        skipped+=1
        continue

    label = check_syntax(code)

    embedding = embedding_model.encode(prompt + reasoning)

    prior = compute_prior_history(embedding)

    feats = extract_features(prompt,code,mean_logprob,prior)

    feats["failed"]=label

    pd.DataFrame([feats]).to_csv(CSV_FILE,mode="a",header=False,index=False)

    history_labels.append(label)

    faiss_index.add(np.array([embedding]).astype("float32"))

    faiss.write_index(faiss_index,FAISS_FILE)

    np.save(LABEL_FILE,np.array(history_labels))


print("Dataset generation finished")
print("Skipped:",skipped)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 946.13it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
 10%|█         | 1/10 [00:36<05:32, 36.92s/it]

In [ ]:
# ==============================
# INSTALL FIRST
# ==============================
# pip install datasets tqdm radon sentence-transformers faiss-cpu pandas numpy torch transformers

import ast
import json
import re
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset
from radon.complexity import cc_visit
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import faiss
import os

torch.set_grad_enabled(False)

# ==============================
# CONFIG
# ==============================

MAX_SAMPLES = 50
MAX_PROMPT_LEN = 400
MAX_NEW_TOKENS = 800
TOP_K_HISTORY = 2

CSV_FILE = "failure_risk_dataset.csv"
FAISS_FILE = "history_index.faiss"
LABEL_FILE = "history_labels.npy"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ==============================
# LOAD MODEL
# ==============================

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "codellama/CodeLlama-7b-hf"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32
).to(DEVICE)

tokenizer.pad_token = tokenizer.eos_token

# ==============================
# DATASET
# ==============================

dataset = load_dataset("code_search_net", "python")["train"]
dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

# ==============================
# EMBEDDING MODEL
# ==============================

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dim = 384

# ==============================
# VECTOR DB
# ==============================

if os.path.exists(FAISS_FILE):
    faiss_index = faiss.read_index(FAISS_FILE)
    history_labels = list(np.load(LABEL_FILE))
else:
    faiss_index = faiss.IndexFlatL2(embedding_dim)
    history_labels = []

# ==============================
# UTILITIES
# ==============================

def clean_code(code: str):
    code = code.replace("```python","").replace("```","")
    return code.strip()


def extract_json(text):

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1:
        try:
            return json.loads(text[start:end+1])
        except:
            return None

    return None


def extract_code(text):

    # python fenced block
    blocks = re.findall(r"```python(.*?)```", text, re.DOTALL)
    if blocks:
        return blocks[-1]

    # generic fenced block
    blocks = re.findall(r"```(.*?)```", text, re.DOTALL)
    if blocks:
        return blocks[-1]

    # fallback: detect function
    lines = text.split("\n")

    code = []
    capture = False

    for l in lines:

        if l.strip().startswith("def "):
            capture = True

        if capture:
            code.append(l)

    return "\n".join(code)


def compute_confidence(mean_logprob):

    return float(np.exp(mean_logprob))


def check_syntax(code):

    if not code.strip():
        return 1

    try:
        compile(code,"<string>","exec")
        return 0
    except:
        return 1


def compute_prior_history(embedding):

    if len(history_labels) < TOP_K_HISTORY:
        return 0.5

    query = np.array([embedding]).astype("float32")

    distances,indices = faiss_index.search(query,TOP_K_HISTORY)

    sims = 1/(1+distances[0])

    labels = np.array(history_labels)[indices[0]]

    prior = np.sum(sims*labels)/np.sum(sims)

    return float(prior)


# ==============================
# FEATURE EXTRACTION
# ==============================

def extract_features(prompt,code,mean_logprob,prior):

    features = {}

    features["prompt_len"] = len(prompt)
    features["code_len"] = len(code)

    try:
        tree = ast.parse(code)
        features["ast_nodes"] = sum(1 for _ in ast.walk(tree))
    except:
        features["ast_nodes"] = 0

    try:
        comp = cc_visit(code)
        features["avg_complexity"] = sum(c.complexity for c in comp)/len(comp) if comp else 0
    except:
        features["avg_complexity"] = 0

    features["mean_logprob"] = mean_logprob
    features["confidence"] = compute_confidence(mean_logprob)
    features["prior_history"] = prior

    return features


# ==============================
# GENERATION
# ==============================

def generate_reasoning_code(prompt,func_name):

    prompt = prompt[:MAX_PROMPT_LEN]

    full_prompt = f"""
You are an expert Python programmer.

Return STRICT JSON ONLY.

Output format:

{{
"reasoning": "short reasoning",
"code": "complete python function"
}}

Write the Python implementation.

Function name: {func_name}

Description:
{prompt}

Remember:
- Only return JSON
- Code must start with def
- No explanations outside JSON
"""

    inputs = tokenizer(full_prompt,return_tensors="pt").to(DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id
    )

    seq = outputs.sequences[0]
    new_tokens = seq[inputs.input_ids.shape[1]:]

    logprobs = []

    for i,score in enumerate(outputs.scores):
        probs = F.log_softmax(score,dim=-1)
        logprobs.append(probs[0,new_tokens[i]].item())

    mean_logprob = float(np.mean(logprobs)) if logprobs else -100

    text = tokenizer.decode(new_tokens,skip_special_tokens=True)

    data = extract_json(text)

    if data and "code" in data:

        reasoning = data.get("reasoning","")
        code = data["code"]

    else:

        reasoning = ""
        code = extract_code(text)

    code = clean_code(code)

    if code.strip() == "..." or len(code.strip()) < 10:
        code = ""

    return reasoning,code,mean_logprob


# ==============================
# CSV INIT
# ==============================

if not os.path.exists(CSV_FILE):

    cols = [
        "prompt_len",
        "code_len",
        "ast_nodes",
        "avg_complexity",
        "mean_logprob",
        "confidence",
        "prior_history",
        "failed"
    ]

    pd.DataFrame(columns=cols).to_csv(CSV_FILE,index=False)

# ==============================
# MAIN LOOP
# ==============================

skipped = 0

for i,item in enumerate(tqdm(dataset)):

    prompt = item["func_documentation_string"]

    if len(prompt) > 600:
        skipped += 1
        continue

    reference = item["func_code_string"]

    try:
        func_name = reference.split("def ")[1].split("(")[0]
    except:
        func_name = "solution"

    try:
        reasoning,code,mean_logprob = generate_reasoning_code(prompt,func_name)
    except Exception as e:
        print("generation failed",e)
        skipped += 1
        continue

    label = check_syntax(code)

    embedding = embedding_model.encode(prompt + " " + reasoning)

    prior = compute_prior_history(embedding)

    feats = extract_features(prompt,code,mean_logprob,prior)

    feats["failed"] = label

    pd.DataFrame([feats]).to_csv(CSV_FILE,mode="a",header=False,index=False)

    history_labels.append(label)

    faiss_index.add(np.array([embedding]).astype("float32"))

    if (i+1)%10==0:

        faiss.write_index(faiss_index,FAISS_FILE)
        np.save(LABEL_FILE,np.array(history_labels))

    if i < 5:

        print("\n--- SAMPLE ---")
        print("Prompt:",prompt[:120])
        print("Code:",code[:200])
        print("Syntax OK:",label==0)
        print("Prior:",prior)

# ==============================
# SAVE
# ==============================

faiss.write_index(faiss_index,FAISS_FILE)
np.save(LABEL_FILE,np.array(history_labels))

print("\nDataset generation complete")
print("Skipped:",skipped)

c:\Users\VINIL\Desktop\Failure_Aware_Agent\Environment\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]